# GENERAL PURPOSE PREDICTION SYSTEM

## IMPORTING

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn import set_config

# This tells Jupyter to render interactive diagrams for our pipelines!
set_config(display="diagram")

print("Libraries imported successfully.")

## Load Data & Define Target

In [ ]:
# 1. Define your test inputs
csv_path = 'your_dataset.csv' # e.g., 'housing_prices.csv'
target_col = 'Target_Column_Name' # e.g., 'Price'
model_save_path = './best_model.joblib'

# 2. Load and prep data
df = pd.read_csv(csv_path)
X_full = df.drop(columns=[target_col])
y = df[target_col]

# 3. Detect types and task
num_cols = X_full.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_full.select_dtypes(include=['object', 'category']).columns.tolist()

is_classification = (y.dtype == 'object' or y.dtype.name == 'category' or y.nunique() < 20)
y_numeric = pd.factorize(y)[0] if is_classification else y

print(f"Task Type: {'Classification' if is_classification else 'Regression'}")
display(df.head())

## Phase 1 - Feature Selection

In [ ]:
# --- Numerical Filter (> 0.5) ---
selected_num_cols = []
if len(num_cols) > 0:
    correlations = X_full[num_cols].corrwith(pd.Series(y_numeric, index=y.index)).abs()
    selected_num_cols = correlations[correlations > 0.5].index.tolist()

# --- Categorical Filter (Mutual Info) ---
selected_cat_cols = []
if len(cat_cols) > 0:
    valid_cat_cols = [col for col in cat_cols if X_full[col].nunique() < (len(X_full) * 0.4)]
    if len(valid_cat_cols) > 0:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_cat_encoded = oe.fit_transform(X_full[valid_cat_cols].astype(str))
        
        if is_classification:
            mi_scores = mutual_info_classif(X_cat_encoded, y_numeric, discrete_features=True, random_state=42)
        else:
            mi_scores = mutual_info_regression(X_cat_encoded, y_numeric, discrete_features=True, random_state=42)
            
        mi_series = pd.Series(mi_scores, index=valid_cat_cols)
        selected_cat_cols = mi_series[mi_series > 0.02].index.tolist()

selected_features = selected_num_cols + selected_cat_cols
X_selected = X_full[selected_features]

print(f"Selected Numerical Features: {selected_num_cols}")
print(f"Selected Categorical Features: {selected_cat_cols}")

## Build Preprocessor Base

In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')), 
        ('scaler', StandardScaler())
    ]), selected_num_cols),
    
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')), 
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), selected_cat_cols)
])

# Display the base preprocessor architecture
preprocessor

## The Model Tournament!

In [ ]:
if is_classification:
    print("Classification task detected. Skipping regression tournament...")
    final_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    final_pipeline.fit(X_selected, y)

else:
    print("🏁 Starting Regression Tournament...\n")
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y_numeric, test_size=0.2, random_state=42)
    
    candidate_models = {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(),
        "Lasso Regression": Lasso(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Support Vector Machine": SVR(),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
        "XGBoost": XGBRegressor(random_state=42)
    }
    
    best_rmse = float('inf')
    best_pipeline = None
    best_model_name = ""
    
    # Let them race!
    for name, model in candidate_models.items():
        pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
        
        # Train
        pipeline.fit(X_train, y_train)
        
        # Predict & Evaluate
        preds = pipeline.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        
        print(f"   ➤ {name} RMSE: {rmse:,.4f}")
        
        if rmse < best_rmse:
            best_rmse = rmse
            best_pipeline = pipeline
            best_model_name = name
            
    print(f"\n TOURNAMENT WINNER: {best_model_name}")
    print(f" Winning RMSE Score: {best_rmse:,.4f}")
    
    # Retrain the champion on 100% of the dataset
    print(f"\nRetraining {best_model_name} on 100% of data for final export...")
    best_pipeline.fit(X_selected, y_numeric)
    final_pipeline = best_pipeline

print("Done!")

## Visualizing the Winner & Exporting

In [ ]:
# 1. Visualize the absolute best model
display(final_pipeline)

# 2. Save to joblib
joblib.dump(final_pipeline, model_save_path)
print(f"\n✅ Winning model saved to {model_save_path}")

# 3. Extract the dropdown categories for the UI
cat_mappings = {}
for col in selected_cat_cols:
    cat_mappings[col] = X_full[col].dropna().astype(str).unique().tolist()

print("\n--- UI Dropdown Data Generated ---")
for key, values in cat_mappings.items():
    print(f"{key}: {values}")